In [ ]:
import pandas as pd
import numpy as np
import joblib
import holidays
import warnings
from google.colab import drive
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Database.xlsx'

sheets = pd.read_excel(file_path, sheet_name=None)
df_raw = pd.concat(sheets.values(), ignore_index=True)

df = df_raw[['REGISTER', 'POLI']].copy()
df.columns = ['Tanggal', 'Poliklinik']
df['Tanggal'] = pd.to_datetime(df['Tanggal'])

def preprocess_poli(data, nama_poli):
    df_temp = data[data['Poliklinik'] == nama_poli].copy()
    df_daily = df_temp.groupby('Tanggal').size().reset_index(name='y')
    df_daily.columns = ['ds', 'y']

    id_holidays = holidays.Indonesia()
    df_daily = df_daily[df_daily['ds'].dt.dayofweek < 5]

    df_daily = df_daily[~df_daily['ds'].apply(lambda x: x in id_holidays)]

    df_daily = df_daily[df_daily['y'] > 5]
    return df_daily.sort_values('ds')

def calculate_metrics(y_true, y_pred, label, poli):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    return {
        'Poliklinik': poli,
        'Model': label,
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'MAPE (%)': round(mape, 2)
    }

daftar_poli = [
    'Poliklinik Penyakit Dalam',
    'Poliklinik Rehabilitasi Medik',
    'Poliklinik Jantung dan Pembuluh Darah',
    'Poliklinik Paru',
    'Poliklinik Umum MCU'
]

summary_results = []

print("=== MEMULAI ANALISIS EVALUASI ARIMA ===")

for poli in daftar_poli:
    print(f"Sedang memproses: {poli}...")

    df_eval = preprocess_poli(df, poli)

    split_idx = int(len(df_eval) * 0.8)
    train = df_eval.iloc[:split_idx]
    test = df_eval.iloc[split_idx:]
    y_test = test['y'].values

    history = [x for x in train['y'].values]
    preds = []

    for t in range(len(test)):
        model_arima = ARIMA(history, order=(7, 1, 1))
        model_fit = model_arima.fit()
        preds.append(model_fit.forecast()[0])
        history.append(y_test[t])

    metrics = calculate_metrics(y_test, np.array(preds), 'ARIMA', poli)
    summary_results.append(metrics)

df_final = pd.DataFrame(summary_results)

rerata_mae = df_final['MAE'].mean()
rerata_mape = df_final['MAPE (%)'].mean()

print("\n" + "="*65)
print("   HASIL EVALUASI MODEL ARIMA SELURUH POLIKLINIK")
print("="*65)
print(df_final[['Poliklinik', 'MAE', 'RMSE', 'MAPE (%)']].to_string(index=False))
print("-" * 65)
print(f"RATA-RATA MAE TOTAL  : {round(rerata_mae, 2)}")
print(f"RATA-RATA MAPE TOTAL : {round(rerata_mape, 2)}%")
print("="*65)

for poli in daftar_poli:
    df_final_train = preprocess_poli(df, poli)

    model_prod = ARIMA(df_final_train['y'].values, order=(7, 1, 1))
    model_fit_prod = model_prod.fit()

    filename = f"model_arima_{poli.replace(' ', '_')}.pkl"
    joblib.dump(model_fit_prod, filename)
    print(f"✅ Model {poli} berhasil di-export sebagai {filename}")

df_final.to_excel('Hasil_Evaluasi_ARIMA_Final.xlsx', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== MEMULAI ANALISIS EVALUASI ARIMA ===
Sedang memproses: Poliklinik Penyakit Dalam...
Sedang memproses: Poliklinik Rehabilitasi Medik...
Sedang memproses: Poliklinik Jantung dan Pembuluh Darah...
Sedang memproses: Poliklinik Paru...
Sedang memproses: Poliklinik Umum MCU...

   HASIL EVALUASI MODEL ARIMA SELURUH POLIKLINIK
                           Poliklinik   MAE  RMSE  MAPE (%)
            Poliklinik Penyakit Dalam 13.75 17.48     23.31
        Poliklinik Rehabilitasi Medik  4.25  5.33     13.61
Poliklinik Jantung dan Pembuluh Darah 11.05 14.00     34.47
                      Poliklinik Paru  5.64  7.47     33.58
                  Poliklinik Umum MCU 14.07 26.97     71.41
-----------------------------------------------------------------
RATA-RATA MAE TOTAL  : 9.75
RATA-RATA MAPE TOTAL : 35.28%
✅ Model Poliklinik Penyakit Dalam berhasil di-export sebagai m